# 🤟 Wasel v4 Pro — Live Sign Language Translator
### MediaPipe Holistic + Self-Training + Real-Time Translation

**3 Steps:**
1. **Cell 1:** Install dependencies
2. **Cell 2:** Record your signs & train the AI
3. **Cell 3:** Launch live translator!

> ⚡ **Zero external dependencies. The AI learns YOUR signs in seconds!**

In [ ]:
#@title 📦 Cell 1: Install & Setup

# IMPORTANT: Pin mediapipe to 0.10.14 — newer versions removed solutions API
print("⏳ Installing dependencies...")
!pip install -q "gradio>=5.0.0" "mediapipe==0.10.14" "scikit-learn>=1.3.0" "opencv-python-headless"

# Restart mediapipe import after install
import importlib
import mediapipe
importlib.reload(mediapipe)

import cv2, pickle, os, logging
import numpy as np
import mediapipe as mp
from collections import deque
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("wasel")

# Verify mediapipe has solutions
assert hasattr(mp, 'solutions'), f"mediapipe {mp.__version__} missing solutions! Need ==0.10.14"
print(f"   ✅ mediapipe: {mp.__version__}")

# ─── MediaPipe Holistic ───
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_features(results):
    """Extract 225 features from MediaPipe Holistic."""
    features = []
    # Pose: 33 × 3 = 99
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 99)
    # Left hand: 21 × 3 = 63
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    # Right hand: 21 × 3 = 63
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    return np.array(features)  # = 225

print("✅ MediaPipe Holistic ready (pose 33 + hands 21×2 = 225 features)")
print("🎉 Proceed to Cell 2!")

In [ ]:
#@title 🎓 Cell 2: Record Signs & Train AI
#@markdown 1. Type sign name → show sign on camera
#@markdown 2. Repeat for 3+ signs
#@markdown 3. Click **Train AI**!

import gradio as gr

training_data = {}
classifier = None
label_encoder = None

def record_frame(sign_name, image):
    if not sign_name or not sign_name.strip():
        return image, "❌ Type a sign name first!"
    if image is None:
        return None, "❌ No camera image!"

    label = sign_name.strip().lower()
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = holistic.process(rgb)
    out = image.copy()

    # Draw skeleton
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

    # Save features
    feat = extract_features(results)
    if label not in training_data:
        training_data[label] = []
    training_data[label].append(feat)

    cv2.putText(out, f"REC: {label} ({len(training_data[label])})", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    status = f"Recording '{label}': {len(training_data[label])} frames\n\nAll signs:\n"
    for k, v in training_data.items():
        status += f"  • {k}: {len(v)} frames\n"
    return out, status

def train_model():
    global classifier, label_encoder
    if len(training_data) < 2:
        return "❌ Record at least 2 different signs!"

    X, y = [], []
    for label, feats in training_data.items():
        for f in feats:
            X.append(f)
            y.append(label)
    X, y = np.array(X), np.array(y)

    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    if len(X) > 10:
        Xtr, Xte, ytr, yte = train_test_split(X, y_enc, test_size=0.2, random_state=42)
        clf.fit(Xtr, ytr)
        acc = accuracy_score(yte, clf.predict(Xte))
    else:
        clf.fit(X, y_enc)
        acc = 1.0

    classifier = clf
    with open('/content/wasel_model.pkl', 'wb') as f:
        pickle.dump((classifier, label_encoder), f)

    return (f"🎉 TRAINED!\n"
            f"   Accuracy: {acc*100:.1f}%\n"
            f"   Signs: {list(label_encoder.classes_)}\n"
            f"   Samples: {len(X)}\n"
            f"   Features: {X.shape[1]}\n\n"
            f"🚀 Now run Cell 3!")

def clear_all():
    global training_data
    training_data = {}
    return "🗑️ Cleared!"

with gr.Blocks(title="Wasel v4 Recorder") as recorder:
    gr.Markdown("## 🎓 Teach the AI Your Signs")
    sign_name = gr.Textbox(label="Sign Name", placeholder="hello, water, help...")
    with gr.Row():
        cam = gr.Image(sources=["webcam"], streaming=True, label="📸 Show Sign")
        preview = gr.Image(label="🦴 Skeleton")
    status = gr.Textbox(label="Status", lines=6, interactive=False)
    with gr.Row():
        train_btn = gr.Button("🧠 Train AI", variant="primary", scale=2)
        clear_btn = gr.Button("🗑️ Clear", variant="stop")
    cam.stream(fn=record_frame, inputs=[sign_name, cam], outputs=[preview, status])
    train_btn.click(fn=train_model, outputs=[status])
    clear_btn.click(fn=clear_all, outputs=[status])

print("🎉 Launching Recorder...")
recorder.launch(share=True, quiet=False)

In [ ]:
#@title 🎥 Cell 3: Live Translator
#@markdown Uses the model you trained in Cell 2!

import gradio as gr

MODEL_PATH = '/content/wasel_model.pkl'
if not os.path.exists(MODEL_PATH):
    raise RuntimeError("❌ Run Cell 2 first and click Train!")

with open(MODEL_PATH, 'rb') as f:
    classifier, label_encoder = pickle.load(f)
print(f"✅ Model: {list(label_encoder.classes_)}")

buf = deque(maxlen=10)

def translate(image):
    if image is None:
        return image
    try:
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        out = image.copy()
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

        feat = extract_features(results)
        buf.append(feat)
        text = "Waiting..."
        if len(buf) >= 3:
            avg = np.mean(np.array(list(buf)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 30.0:
                text = f"{label_encoder.inverse_transform([idx])[0].upper()} ({conf:.1f}%)"
        cv2.putText(out, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        return out
    except Exception as e:
        return image

demo = gr.Interface(
    fn=translate,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸"),
    outputs=gr.Image(label="🤟 Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Live PSL Translator",
    description="Show your signs — AI translates in real-time!",
)
print("🎉 Launching...")
demo.launch(share=True, quiet=False)